In [ ]:
import pandas as pd
import requests
import io

# 1. Daten via OSF API laden
osf_project_id = "dm37b"  # Deine OSF-Projekt-ID
api_url = f"https://api.osf.io/v2/nodes/{osf_project_id}/files/osfstorage/"

print("Verbinde mit OSF und suche nach Datensätzen...")
response = requests.get(api_url)
files_data = response.json()

dataframes = []

# Alle Dateien im OSF-Ordner durchgehen
for file in files_data.get('data', []):
    file_name = file['attributes']['name']
    
    # Wir wollen nur die .csv Dateien einlesen
    if file_name.endswith('.csv'):
        download_url = file['links']['download']
        file_content = requests.get(download_url).content
        
        # CSV in einen pandas DataFrame laden und zur Liste hinzufügen
        df = pd.read_csv(io.StringIO(file_content.decode('utf-8')))
        dataframes.append(df)

# Alle einzelnen Datensätze zu einer großen Tabelle zusammenfügen
raw_data = pd.concat(dataframes, ignore_index=True)
print(f"Erfolgreich {len(dataframes)} Datensätze geladen und zusammengefügt.")

# 2. Preprocessing (Datenreinigung)

# A. Nur die Test-Trials behalten (Fixationskreuze, Instruktionen und Lernphase filtern wir heraus)
clean_data = raw_data[raw_data['task'] == 'test_trial'].copy()

# B. Die Spalte 'correct' (True/False) für die Auswertung in Zahlen (1 = richtig, 0 = falsch) umwandeln
clean_data['accuracy'] = clean_data['correct'].astype(int)

# C. Die Spalte 'study_position' säubern
# "Neue" Wörter haben hier den Wert 'N/A' (Not Applicable). Das stört bei numerischen Analysen.
# Wir ersetzen 'N/A' durch den Wert -1 (als Kennzeichnung für "war nicht in der Lernliste")
clean_data['study_position'] = clean_data['study_position'].replace('N/A', -1)
# Jetzt können wir die Spalte sicher in Zahlen umwandeln
clean_data['study_position'] = pd.to_numeric(clean_data['study_position'])

print(f"Datenreinigung abgeschlossen. Es verbleiben {len(clean_data)} Test-Trials für die Analyse.")
# 3. Definition der Variablen und Aggregation

# Unabhängige Variable (UV): 'study_position' (Position des Wortes in der Lernliste, 1 bis 15)
# Abhängige Variable (AV): 'accuracy' (Mittlere Trefferrate, 0.0 bis 1.0)

# Wir betrachten für diesen Effekt nur die echten Lern-Wörter (Targets)
targets_data = clean_data[clean_data['condition'] == 'old'].copy()

# A. Aggregation auf Item-Ebene (Perfekt für das Diagramm)
# Hier berechnen wir: Wie gut wurde das Wort auf Position 1 im Durchschnitt über ALLE Personen erinnert?
position_accuracy = targets_data.groupby('study_position')['accuracy'].mean().reset_index()

# B. Aggregation auf Personen-Ebene (Wichtig für spätere statistische Tests)
# Hier berechnen wir: Wie gut hat jede einzelne Versuchsperson (Subject) bei jeder Position abgeschnitten?
subject_position_accuracy = targets_data.groupby(['subject', 'study_position'])['accuracy'].mean().reset_index()

# C. Kurzer Check zur Gesamtreaktionszeit (als Bonus-AV)
mean_rt = targets_data['rt'].mean()

print("Daten erfolgreich aggregiert!")
print(f"Die durchschnittliche Reaktionszeit lag bei {mean_rt:.2f} Millisekunden.")
print("\nHier ist eine kleine Vorschau der Trefferrate (accuracy) für die ersten Listenpositionen:")
print(position_accuracy.head())
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Deskriptive Statistik: Tabelle erstellen
# Wir vergleichen, wie gut "alte" Wörter (Targets) und "neue" Wörter (Distraktoren) erkannt wurden
desc_stats = clean_data.groupby('condition')['accuracy'].agg(['mean', 'std', 'count']).reset_index()
desc_stats.rename(columns={'mean': 'Trefferrate (Mittelwert)', 'std': 'Standardabweichung', 'count': 'Anzahl Trials'}, inplace=True)

print("--- Deskriptive Statistik ---")
print(desc_stats)
print("\n")

# 2. Visualisierung: Die Primacy-Recency-Kurve zeichnen
# Wir nutzen den seaborn-Style für ein sauberes, wissenschaftliches Design
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

# Ein Liniendiagramm mit Punkten für jede Listenposition
plot = sns.lineplot(
    data=position_accuracy, 
    x='study_position', 
    y='accuracy', 
    marker='o', 
    markersize=8, 
    linewidth=2.5,
    color='#2c3e50'
)

# Achsen und Titel hübsch machen
plt.title('Erinnerungsleistung in Abhängigkeit der Listenposition', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Position in der Lernliste (1 = Erstes Wort, 15 = Letztes Wort)', fontsize=12)
plt.ylabel('Trefferrate (Accuracy)', fontsize=12)

# Y-Achse von 0 bis 1 (0% bis 100%) festlegen, damit das Diagramm nicht verzerrt
plt.ylim(0, 1.05)

# X-Achse so einstellen, dass wirklich jede Zahl von 1 bis 15 als Tick angezeigt wird
plt.xticks(range(1, 16))

# Diagramm anzeigen
plt.tight_layout()
plt.show()
import scipy.stats as stats

print("--- Prüfung der statistischen Voraussetzungen ---")

# 1. Normalverteilung prüfen (Shapiro-Wilk-Test)
# Wir prüfen hier beispielhaft die aggregierten Daten über die Positionen
stat_sw, p_sw = stats.shapiro(position_accuracy['accuracy'])

print("\n1. Normalverteilung (Shapiro-Wilk-Test):")
if p_sw > 0.05:
    print(f"p = {p_sw:.3f} -> Alles im grünen Bereich! Die Daten weichen nicht signifikant von einer Normalverteilung ab.")
else:
    print(f"p = {p_sw:.3f} -> ACHTUNG: Die Daten sind NICHT normalverteilt.")

# 2. Varianzhomogenität prüfen (Levene-Test)
# Wir vergleichen die Streuung der Trefferraten für die ersten drei (Primacy) und die mittleren drei Positionen
primacy_data = subject_position_accuracy[subject_position_accuracy['study_position'].isin([1, 2, 3])]['accuracy']
middle_data = subject_position_accuracy[subject_position_accuracy['study_position'].isin([7, 8, 9])]['accuracy']

# Levene-Test verlangt, dass wir die Gruppen einzeln übergeben (dropna entfernt evtl. leere Einträge)
stat_lev, p_lev = stats.levene(primacy_data.dropna(), middle_data.dropna())

print("\n2. Varianzhomogenität (Levene-Test zwischen Anfang und Mitte der Liste):")
if p_lev > 0.05:
    print(f"p = {p_lev:.3f} -> Alles im grünen Bereich! Die Gruppen streuen ähnlich stark.")
else:
    print(f"p = {p_lev:.3f} -> ACHTUNG: Die Varianzen (Streuungen) sind signifikant unterschiedlich.")

# 3. Kommentar zum weiteren Vorgehen (Was tun bei Verletzung?)
print("\n--- Was tun, wenn Voraussetzungen verletzt sind? ---")
print("Da unsere Trefferrate bei 100% (1.0) gedeckelt ist, verletzen wir häufig die Normalverteilung.")
print("Das ist kein Grund zur Panik! Wir können auf 'robuste' (non-parametrische) Tests ausweichen")
print("oder - was wir in Task 6 tun werden - ein Multilevel-Modell (MLM) nutzen, das extra für solche Daten gemacht ist.")
import numpy as np
import scipy.stats as stats

# 1. Daten in drei Zonen einteilen: Anfang (Primacy), Mitte (Vergessen), Ende (Recency)
def get_zone_accuracy(positions):
    # Berechnet die durchschnittliche Trefferrate pro Person für bestimmte Positionen
    zone_data = targets_data[targets_data['study_position'].isin(positions)]
    return zone_data.groupby('subject')['accuracy'].mean()

# Positionen 1-3 (Anfang), 7-9 (Mitte), 13-15 (Ende - bei 15 Lern-Wörtern)
primacy_acc = get_zone_accuracy([1, 2, 3])
middle_acc = get_zone_accuracy([7, 8, 9])
recency_acc = get_zone_accuracy([13, 14, 15])

# Sicherstellen, dass wir nur Personen vergleichen, die in allen Zonen Daten haben
common_subjects = primacy_acc.index.intersection(middle_acc.index).intersection(recency_acc.index)
primacy_acc = primacy_acc[common_subjects]
middle_acc = middle_acc[common_subjects]
recency_acc = recency_acc[common_subjects]

# 2. Statistische Tests (Gepaarte t-Tests)
# A. Primacy-Effekt prüfen: Anfang vs. Mitte
t_prim, p_prim = stats.ttest_rel(primacy_acc, middle_acc)

# B. Recency-Effekt prüfen: Ende vs. Mitte
t_rec, p_rec = stats.ttest_rel(recency_acc, middle_acc)

# 3. Effektstärke (Cohen's d für verbundene Stichproben) berechnen
def cohens_d(x, y):
    diff = x - y
    return np.mean(diff) / np.std(diff, ddof=1)

d_prim = cohens_d(primacy_acc, middle_acc)
d_rec = cohens_d(recency_acc, middle_acc)

# 4. Ergebnisse ausgeben
print("--- Ergebnisse der Gruppenvergleiche ---")
print(f"Primacy-Effekt (Anfang vs. Mitte): t = {t_prim:.2f}, p = {p_prim:.4f}, Cohen's d = {d_prim:.2f}")
print(f"Recency-Effekt (Ende vs. Mitte):   t = {t_rec:.2f}, p = {p_rec:.4f}, Cohen's d = {d_rec:.2f}")

import statsmodels.formula.api as smf
import pandas as pd

# 1. Daten für das Modell vorbereiten: Wir erstellen eine neue Spalte 'zone'
def assign_zone(pos):
    if pos in [1, 2, 3]: return '1_Primacy'
    elif pos in [7, 8, 9]: return '2_Middle'
    elif pos in [13, 14, 15]: return '3_Recency'
    else: return 'Other'

# Wir nutzen unsere sauberen Target-Daten aus Task 2
mlm_data = targets_data.copy()
mlm_data['zone'] = mlm_data['study_position'].apply(assign_zone)

# Wir filtern die "Other"-Positionen heraus, um den reinen Effekt zu sehen
mlm_data = mlm_data[mlm_data['zone'] != 'Other']

# 2. Das Multilevel-Modell definieren und berechnen
# Formel-Logik: Trefferrate (accuracy) wird vorhergesagt durch die Zone (C(zone) = kategorisch).
# groups=mlm_data['subject'] bedeutet: Jede Versuchsperson bekommt ihren eigenen Startwert (Random Intercept).
model = smf.mixedlm("accuracy ~ C(zone)", mlm_data, groups=mlm_data["subject"])
result = model.fit()

# 3. Ergebnisse ausgeben
print(result.summary())
